In [0]:
def update_watermarks(catalog, schema_name, watermark_updates):
    """
    Batch updates watermarks for multiple tables in a single MERGE.
    """
    watermark_table = f"{catalog}.{schema_name}.etl_watermark"

    rows = [f"SELECT '{t}' AS table_name, CAST('{ts}' AS TIMESTAMP) AS last_modified, current_timestamp() AS updated_at" for t, ts in watermark_updates]
    union_query = " UNION ALL ".join(rows)

    spark.sql(f"""
        MERGE INTO {watermark_table} AS target
        USING ({union_query}) AS source
        ON target.table_name = source.table_name
        WHEN MATCHED THEN UPDATE SET
            target.last_modified = source.last_modified,
            target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT (table_name, last_modified, updated_at)
            VALUES (source.table_name, source.last_modified, source.updated_at)
    """)

    logger.info(f"Batch watermark update: {len(watermark_updates)} tables")